# Milestone 2 — Data Understanding & Cleaning
**Dataset:** Kaggle [`giovamata/airlinedelaycauses`](https://www.kaggle.com/datasets/giovamata/airlinedelaycauses) (`DelayedFlights.csv`) — US domestic flights 2008 (delayed flights subset; note: not all flights, so delay rates here are NOT national on-time rates)
**Goal:** understand the data, clean it with justified decisions, and prove it is now trustworthy.

Sections: 1. Load · 2. Overview & data dictionary · 3. Missing values · 4. Duplicates · 5. Before stats · 6. Cleaning decisions · 7. Outliers · 8. After stats · 9. Before/after · 10. Remaining issues · 11. Save

In [ ]:
import pandas as pd, numpy as np
from io import StringIO
pd.set_option('display.max_columns', 50)

# pip install kagglehub   (run once if not installed)
import os, kagglehub

try:
    path = kagglehub.dataset_download("giovamata/airlinedelaycauses")   # downloads/caches from Kaggle
    df = pd.read_csv(os.path.join(path, "DelayedFlights.csv"))
    # The file has a leftover pandas index column ('Unnamed: 0'); it is not data and would hide duplicates.
    df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed')])
except Exception as e:
    # Fallback: the 5-row sample so the notebook still runs end-to-end
    print('Could not load from Kaggle (', e, ') - using the 5-row sample.')
    sample = '''Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
2008,1,3,4,2003,1955,2211,2225,WN,335,N712SW,128,150,116,-14,8,IAD,TPA,810,4,8,0,N,0,,,,,
2008,1,3,4,754,735,1002,1000,WN,3231,N772SW,128,145,113,2,19,IAD,TPA,810,5,10,0,N,0,,,,,
2008,1,3,4,628,620,804,750,WN,448,N428WN,96,90,76,14,8,IND,BWI,515,3,17,0,N,0,,,,,
2008,1,3,4,1829,1755,1959,1925,WN,3920,N464WN,90,90,77,34,34,IND,BWI,515,3,10,0,N,0,2,0,0,0,32
2008,1,3,4,1940,1915,2121,2110,WN,378,N726SW,101,115,87,11,25,IND,JAX,688,4,10,0,N,0,,,,,'''
    df = pd.read_csv(StringIO(sample))

df_raw = df.copy()   # untouched copy for before/after comparison
df.head()

## 2. Dataset overview & data dictionary

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
dictionary = pd.DataFrame({
 'column': ['Year','Month','DayofMonth','DayOfWeek','DepTime','CRSDepTime','ArrTime','CRSArrTime','UniqueCarrier','FlightNum','TailNum',
            'ActualElapsedTime','CRSElapsedTime','AirTime','ArrDelay','DepDelay','Origin','Dest','Distance','TaxiIn','TaxiOut',
            'Cancelled','CancellationCode','Diverted','CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay'],
 'meaning': ['Year','Month (1-12)','Day of month','Day of week (1=Mon..7=Sun)','Actual departure time (hhmm)','Scheduled departure time (hhmm)',
             'Actual arrival time (hhmm)','Scheduled arrival time (hhmm)','Carrier code','Flight number','Aircraft tail number',
             'Actual elapsed time (min)','Scheduled elapsed time (min)','Time in air (min)','Arrival delay (min, negative = early)',
             'Departure delay (min)','Origin airport','Destination airport','Distance (miles)','Taxi-in time (min)','Taxi-out time (min)',
             'Flight cancelled (1/0)','Reason: A carrier, B weather, C NAS, D security','Flight diverted (1/0)',
             'Delay due to carrier (min)','Delay due to weather (min)','Delay due to NAS (min)','Delay due to security (min)','Delay due to late aircraft (min)']})
dictionary['dtype'] = dictionary['column'].map(df.dtypes.astype(str))
dictionary

In [ ]:
df.describe().T

In [ ]:
print('Cancelled:', df['Cancelled'].mean().round(4), '| Diverted:', df['Diverted'].mean().round(4))
print(df['CancellationCode'].value_counts(dropna=False))
print('Carriers:', df['UniqueCarrier'].nunique(), '| Origins:', df['Origin'].nunique(), '| Dests:', df['Dest'].nunique())

## 3. Missing values analysis

In [ ]:
miss = pd.DataFrame({'missing': df.isna().sum(), 'pct': (df.isna().mean()*100).round(2)})
miss[miss.missing>0].sort_values('missing', ascending=False)

In [ ]:
# WHY are values missing? Check the pattern instead of guessing.
time_cols = ['DepTime','ArrTime','ActualElapsedTime','AirTime','ArrDelay','DepDelay','TaxiIn','TaxiOut']
print('Missing time/delay values by Cancelled / Diverted flag:')
print(df.groupby(['Cancelled','Diverted'])[time_cols].apply(lambda g: g.isna().mean().round(2)))

delay_causes = ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
print('\nDelay-cause columns: missing share when ArrDelay >= 15 vs < 15')
print(df.groupby(df['ArrDelay'] >= 15)[delay_causes].apply(lambda g: g.isna().mean().round(2)))

**Findings (explain in demo):**
- Time/delay columns are missing mainly for **cancelled** (and some diverted) flights → *structural* missingness: the flight never operated, so the value genuinely doesn't exist.
- Delay-cause columns are only filled when a flight arrives **≥ 15 min late**; otherwise blank → missing means "no reportable delay".

## 4. Duplicate analysis

In [ ]:
full_dups = df.duplicated().sum()
key = ['Year','Month','DayofMonth','UniqueCarrier','FlightNum','Origin','Dest','CRSDepTime']
key_dups = df.duplicated(subset=key, keep=False).sum()
print('Fully identical rows :', full_dups)
print('Rows sharing flight key (carrier+flight+route+date+time):', key_dups)
df[df.duplicated(subset=key, keep=False)].sort_values(key).head(10)

## 5. Summary statistics — BEFORE cleaning

In [ ]:
num_cols = ['DepDelay','ArrDelay','Distance','AirTime','ActualElapsedTime','TaxiIn','TaxiOut']
before_stats = df_raw[num_cols].describe().T
before_snapshot = {'Rows': len(df_raw), 'Missing values': int(df_raw.isna().sum().sum()), 'Duplicate rows': int(df_raw.duplicated().sum())}
before_stats

## 6. Cleaning decisions

### Decision 1 — Drop exact duplicate rows

In [ ]:
# Decision: dropped fully identical rows.
# Reason: an identical record is the same flight counted twice and would bias counts and averages.
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print('Dropped', n0 - len(df), 'duplicate rows')

### Decision 2 — Fix data types

In [ ]:
# Decision: cast flag columns to int8 and categorical columns to 'category'; nullable Int64 for time fields with NaN.
# Reason: saves memory, prevents accidental math on codes, and keeps NaN support without turning ints into floats.
for c in ['Cancelled','Diverted']:
    df[c] = df[c].astype('int8')
for c in ['UniqueCarrier','Origin','Dest','CancellationCode']:
    df[c] = df[c].astype('category')
int_cols = ['DepTime','ArrTime','ActualElapsedTime','AirTime','ArrDelay','DepDelay','TaxiIn','TaxiOut'] + delay_causes
for c in int_cols:
    df[c] = df[c].astype('Int64')

# Decision: built a proper FlightDate column.
# Reason: Year/Month/DayofMonth separately are awkward for time-based analysis in Module 3.
df['FlightDate'] = pd.to_datetime(dict(year=df.Year, month=df.Month, day=df.DayofMonth))
df.dtypes

### Decision 3 — Delay-cause columns: fill with 0 for flights that operated and were not delayed ≥ 15 min

In [ ]:
# Decision: filled delay-cause NaNs with 0 ONLY for non-cancelled, non-diverted flights.
# Reason: blank means 'no reportable delay' (cause breakdown is only recorded for delays >= 15 min), so 0 is the true value.
# Cancelled/diverted flights stay NaN - they never completed, so a delay of 0 would be misleading.
mask = (df.Cancelled == 0) & (df.Diverted == 0)
df.loc[mask, delay_causes] = df.loc[mask, delay_causes].fillna(0)
print(df[delay_causes].isna().sum())

### Decision 4 — Cancelled / diverted flights: keep the rows, do NOT impute times

In [ ]:
# Decision: kept cancelled & diverted rows with NaN in time/delay columns; did not fill or drop.
# Reason: cancellations are real events (valuable for analysis). Imputing a median arrival time for a flight that
# never arrived would invent data. Analyses of delays will filter to Cancelled == 0 explicitly.
print('Cancelled rows kept:', int(df.Cancelled.sum()), '| Diverted rows kept:', int(df.Diverted.sum()))
# CancellationCode 'N' / blank for non-cancelled is valid, not an error
# Blank / 'N' CancellationCode on non-cancelled flights is valid (no cancellation), not an error.
print(df['CancellationCode'].value_counts(dropna=False))

### Decision 5 — Validity checks (logical consistency)

In [ ]:
checks = {
 'Distance <= 0': (df.Distance <= 0).sum(),
 'AirTime > ActualElapsedTime': (df.AirTime > df.ActualElapsedTime).sum(),
 'Origin == Dest': (df.Origin.astype(str) == df.Dest.astype(str)).sum(),
 'Month outside 1-12': (~df.Month.between(1,12)).sum(),
 'DayOfWeek outside 1-7': (~df.DayOfWeek.between(1,7)).sum(),
 'Cancelled but has DepTime': ((df.Cancelled==1) & df.DepTime.notna()).sum(),
}
pd.Series(checks, name='violations')

In [ ]:
# Decision: drop only rows that are logically impossible (AirTime > ActualElapsedTime, Distance <= 0).
# Reason: these cannot be real; every other check is just reported.
bad = (df.Distance <= 0) | (df.AirTime.fillna(0) > df.ActualElapsedTime.fillna(df.AirTime.fillna(0)))
print('Dropping', int(bad.sum()), 'impossible rows')
df = df.loc[~bad].reset_index(drop=True)

## 7. Outlier investigation (investigate, don't delete by default)

In [ ]:
def iqr_outliers(s):
    s = s.dropna().astype(float)
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    lo, hi = q1 - 1.5*i, q3 + 1.5*i
    return pd.Series({'lower': lo, 'upper': hi, 'n_outliers': int(((s<lo)|(s>hi)).sum()), 'max': s.max(), 'min': s.min()})
pd.DataFrame({c: iqr_outliers(df[c]) for c in ['DepDelay','ArrDelay','Distance','AirTime','TaxiIn','TaxiOut']}).T

In [ ]:
# Look at the most extreme delays before deciding anything
df.loc[df.ArrDelay.notna()].nlargest(10, 'ArrDelay')[['FlightDate','UniqueCarrier','FlightNum','Origin','Dest','DepDelay','ArrDelay',
                                                      'CarrierDelay','WeatherDelay','NASDelay','LateAircraftDelay']]

**Decision: outliers kept, not removed or capped.** Delays of several hours are real events (weather, mechanical, cascading late aircraft) and the cause columns support them. Deleting them would hide the true delay picture. Long `Distance` values are real long-haul routes.
Instead, we add a flag so later milestones can include/exclude them explicitly.

In [ ]:
# Decision: added flag column instead of altering values.
df['ExtremeDelay'] = (df.ArrDelay.fillna(0) > 180).astype('int8')   # >3h late
df['ExtremeDelay'].sum()

## 8. Summary statistics — AFTER cleaning

In [ ]:
after_stats = df[num_cols].astype(float).describe().T
after_stats

## 9. Before / After comparison

In [ ]:
after_snapshot = {'Rows': len(df), 'Missing values': int(df.drop(columns=['FlightDate','ExtremeDelay']).isna().sum().sum()), 'Duplicate rows': int(df.duplicated().sum())}
snap = pd.DataFrame({'Before': before_snapshot, 'After': after_snapshot}); snap['Change'] = snap.After - snap.Before
display(snap)
display((after_stats[['mean','50%','max']] - before_stats[['mean','50%','max']]).round(2).add_suffix(' (after-before)'))

## 10. Remaining data quality issues (not fixed, and why)

| Issue | Why left as is |
|---|---|
| Remaining NaN in time/delay columns for cancelled & diverted flights | Structural, the flight never completed; imputing would invent data |
| Extreme delays (> 3h) | Real events; flagged with `ExtremeDelay` rather than removed |
| `DepTime`/`ArrTime` stored as hhmm integers (e.g. 754 = 07:54) | Convert to proper times in Module 3 feature engineering |
| Possible unknown/placeholder `TailNum` values | Not needed for current analysis; check before using tail-level analysis |
| Possible same-key flights that are not exact duplicates | Reported in section 4, kept because they differ in other fields |

## 11. Save the clean dataset

In [ ]:
df.to_csv('flights_2008_clean.csv', index=False)
print('Saved:', df.shape)
# Then: git add flights_2008_clean.csv milestone_2.ipynb && git commit -m 'Milestone 2: cleaning' && git push